In [5]:
import os
import math
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from collections import Counter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
documents = [
    "Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니다.",
    "자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.",
    "벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색하는 시스템입니다.",
    "GPT-4는 OpenAI가 개발한 대규모 언어 모델로 다양한 작업을 수행합니다.",
    "RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM 응답을 개선합니다.",
    "FastAPI는 Python으로 빠른 웹 API를 구축하기 위한 프레임워크입니다.",
    "트랜스포머 아키텍처는 어텐션 메커니즘을 활용한 딥러닝 모델 구조입니다.",
    "FAISS는 Facebook AI가 개발한 효율적인 유사도 검색 라이브러리입니다.",
    "프롬프트 엔지니어링은 LLM에 효과적인 입력을 설계하는 기술입니다.",
    "임베딩은 텍스트를 수치 벡터로 변환하여 의미적 유사성을 측정할 수 있게 합니다.",
]


In [6]:
# 파이썬은 뭔가요? -> Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니다.
#                     파이썬 ~~~

In [7]:
doc_embeddings = embeddings.embed_documents(documents)
len(doc_embeddings), len(doc_embeddings[0])

(10, 1536)

In [45]:
doc_embeddings.shape

AttributeError: 'list' object has no attribute 'shape'

In [9]:
import numpy as np
np.array(doc_embeddings).shape

(10, 1536)

In [10]:
np.array(doc_embeddings)

array([[ 0.00657654,  0.03048706, -0.02264404, ..., -0.05459595,
        -0.01644897,  0.00080776],
       [ 0.0022068 ,  0.00142765, -0.01383209, ..., -0.02345276,
         0.01748657, -0.01129913],
       [-0.02809143, -0.00685883, -0.01005554, ..., -0.01413727,
         0.04055786, -0.01039124],
       ...,
       [-0.02554321, -0.00035667, -0.01853943, ...,  0.03927612,
        -0.04629517, -0.01712036],
       [ 0.0340271 ,  0.0463562 , -0.02189636, ...,  0.00666428,
         0.02731323, -0.01059723],
       [-0.01882935,  0.0411377 ,  0.00099373, ..., -0.01400757,
         0.00837708,  0.00334358]], shape=(10, 1536))

In [11]:
# !pip install sentence-transformers

In [12]:
from sentence_transformers import SentenceTransformer

In [13]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embedding = embedding_model.encode(documents)
embedding.shape

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12607.27it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(10, 384)

In [14]:
embedding

array([[-0.02818313,  0.0231517 , -0.01332182, ...,  0.0944027 ,
         0.06264345, -0.00686624],
       [ 0.004331  ,  0.02231261,  0.06852964, ...,  0.06287044,
         0.03236052,  0.01451775],
       [ 0.04224589,  0.06530681,  0.02390072, ...,  0.01747441,
        -0.08692616,  0.02379499],
       ...,
       [-0.03313583,  0.02055397,  0.00575815, ...,  0.07556441,
        -0.08282115, -0.06177334],
       [ 0.09051043, -0.00739944,  0.04021924, ..., -0.00768866,
        -0.11420643, -0.01269382],
       [ 0.01094473,  0.08185509,  0.03489939, ...,  0.01663295,
        -0.08264519,  0.02735936]], shape=(10, 384), dtype=float32)

In [15]:
# 파이썬 뭔가요?  -> Python 은 ~~~
# ABCDEF 뭔가요 ABCDEF ?  -> abcde 뭔가요?

In [16]:
def keyword_search(query, docs, top_k=3):
    query_tokens = set(query.lower().split())
    scores = []
    for i, doc in enumerate(docs):
        doc_tokens = set(doc.lower().split())
        overlap = len(query_tokens & doc_tokens)
        scores.append((i, overlap))
    
    # scores = [(0, 3), (1, 2), (2, 5) ...]
    scores.sort(key=lambda x:x[1], reverse=True)    # [(2, 5), (0, 3), (1, 2) ... ]
    return scores[:top_k]

In [17]:
results = keyword_search("Python 프로그래밍 언어", documents)
print(results)
for idx, scores in results:
    print(f"[{idx}] overlap = {scores} | {documents[idx][:30]} ") 

[(0, 1), (3, 1), (1, 0)]
[0] overlap = 1 | Python은 데이터 과학과 머신러닝에 널리 사용되는  
[3] overlap = 1 | GPT-4는 OpenAI가 개발한 대규모 언어 모델로  
[1] overlap = 0 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 


In [18]:
def vector_search(query, docs, doc_embs, top_k=3):
    q_emb = np.array(embeddings.embed_query(query))
    similarities = np.dot(doc_embs, q_emb) / (np.linalg.norm(doc_embs, axis=1) * np.linalg.norm(q_emb))
    top_indices = similarities.argsort()[::-1][:top_k]
    return [(i, similarities[i]) for i in top_indices]

results = vector_search("Python 프로그래밍 언어", documents, np.array(doc_embeddings), top_k=3)

In [19]:
results
for idx, scores in results:
    print(f"[{idx}] similarity = {scores} | {documents[idx][:30]} ")

[0] similarity = 0.5528473798288958 | Python은 데이터 과학과 머신러닝에 널리 사용되는  
[1] similarity = 0.37179235223584567 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 
[3] similarity = 0.360448283229595 | GPT-4는 OpenAI가 개발한 대규모 언어 모델로  


In [20]:
results = vector_search('FAISS', documents, np.array(doc_embeddings), top_k=3)

In [21]:
for idx, scores in results:
    print(f"[{idx}] similarity = {scores} | {documents[idx][:30]} ")

[7] similarity = 0.5278809597501696 | FAISS는 Facebook AI가 개발한 효율적인 유 
[4] similarity = 0.19233440678444033 | RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM 
[5] similarity = 0.09642334637561256 | FastAPI는 Python으로 빠른 웹 API를 구축 


In [22]:
def overlap_rate(keyword_results, vector_results):
    
    kw_ids = set(idx for idx, _ in keyword_results)
    vec_ids = set(idx for idx, _ in vector_results)
    
    overlap = kw_ids & vec_ids  # set(0,1,3,4,5,7)
    union = kw_ids | vec_ids    # set()
    return len(overlap) / len(union)
    

In [23]:
for query in ['Python 프로그래밍', '딥러닝 모델', 'FAISS 라이브러리']:
    kw = keyword_search(query, documents, top_k=5)
    vec = vector_search(query, documents, np.array(doc_embeddings), top_k=5)
    rate = overlap_rate(kw, vec)
    print(f"{query} overlap : {rate}")

Python 프로그래밍 overlap : 0.6666666666666666
딥러닝 모델 overlap : 0.42857142857142855
FAISS 라이브러리 overlap : 0.25


In [24]:
kw

[(0, 0), (1, 0), (2, 0), (3, 0), (4, 0)]

In [25]:
vec

[(np.int64(7), np.float64(0.6645911847634929)),
 (np.int64(4), np.float64(0.2495465016632106)),
 (np.int64(2), np.float64(0.21776238835168313)),
 (np.int64(8), np.float64(0.2022434508043166)),
 (np.int64(6), np.float64(0.1956163208615188))]

In [26]:
def simple_hybrid(query, docs, doc_embs, top_k=3):
    kw = keyword_search(query, docs, top_k = len(docs))
    vec = vector_search(query, docs, np.array(doc_embs), top_k= len(docs))
    
    kw_scores = {idx : score for idx, score in kw}
    vec_scores = {idx : score for idx, score in vec}
    
    kw_max = max(kw_scores.values()) or 1
    vec_max = max(vec_scores.values()) or 1
    
    combined = {}
    
    for idx in range(len(docs)):
        kw_score = kw_scores.get(idx, 0) / kw_max
        vec_score = vec_scores.get(idx, 0) / vec_max
        
        combined[idx] = kw_score + vec_score
        
    ranked = sorted(combined.items(), key=lambda x:x[1], reverse=True)
    return ranked[:top_k]
    
#     kw : [(0, 100), (1, 50), (2, 30), ..... (100, 0)] -> [(0, 1), (1, 0.5), (2, 0.3) ...]
#     vec : [(0, 1),  (1, 0.5), (2, 0.9)...]            -> [(0, 1), (1, 0.5), (2, 0.9) ...]

In [27]:
for query in ['Python 프로그래밍 언어', '딥러닝 모델 구조', 'FAISS']:
    results = simple_hybrid(query, documents, np.array(doc_embeddings), top_k=3)
    
#     print(results)
    for idx, score in results:
        print(f"[{idx}] {score} | {documents[idx][:30]}")
        
    print("----------------------------------------------------")

[0] 2.0 | Python은 데이터 과학과 머신러닝에 널리 사용되는 
[3] 1.6519851524685754 | GPT-4는 OpenAI가 개발한 대규모 언어 모델로 
[1] 0.672504502690985 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고
----------------------------------------------------
[6] 2.0 | 트랜스포머 아키텍처는 어텐션 메커니즘을 활용한 딥러닝 
[3] 0.4905194025659972 | GPT-4는 OpenAI가 개발한 대규모 언어 모델로 
[0] 0.4187993403594779 | Python은 데이터 과학과 머신러닝에 널리 사용되는 
----------------------------------------------------
[7] 1.0 | FAISS는 Facebook AI가 개발한 효율적인 유
[4] 0.36435185477321724 | RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM
[5] 0.18266115607057865 | FastAPI는 Python으로 빠른 웹 API를 구축
----------------------------------------------------


###  TF : f(t, d) / | d |    (f(t, d) : document에 나온 t의 개수, |d| : d 의 길이)
###  IDF : log(N / (t가 등장한 문서의 개수)) (N : 전체 문서수)

In [28]:
import math
class TFIDF:
    def __init__(self, documents):
        self.docs = documents
        self.tokenized = [doc.lower().split() for doc in documents]
        self.N = len(documents)
        self.df = {}
        for tokens in self.tokenized:
            for t in set(tokens):
                self.df[t] = self.df.get(t,0) + 1
                
    def tf(self, term, doc_tokens):
        return doc_tokens.count(term) / len(doc_tokens)
    
    def idf(self, term):
        return math.log(self.N / self.df.get(term, 1))
    
    def score(self, query, doc_idx):
        tokens = self.tokenized[doc_idx]
        return sum(self.tf(t, tokens) * self.idf(t) for t in query.lower().split())
    
    def search(self, query, top_k=3):
        scores = [(i, self.score(query, i)) for i in range(self.N)]
        return sorted(scores, key=lambda x: x[1], reverse=True)[:top_k]

In [29]:
tfidf = TFIDF(documents)
for idx, score in tfidf.search('Python 프로그래밍'):
    print(f"[{idx}] {score} | {documents[idx][:30]} ")

[0] 0.28782313662425574 | Python은 데이터 과학과 머신러닝에 널리 사용되는  
[1] 0.0 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 
[2] 0.0 | 벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고  


### k1, b

In [30]:
class BM25:
    def __init__(self, documents, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.docs = documents
        self.tokenized = [doc.lower().split() for doc in documents]
        self.N = len(documents)
        self.avgdl = sum(len(d) for d in self.tokenized) / self.N
        self.df = {}
        for tokens in self.tokenized:
            for t in set(tokens):
                self.df[t] = self.df.get(t, 0) + 1
                
    def idf(self, term):
        df = self.df.get(term, 0)
        return math.log((self.N - df + 0.5) / (df + 0.5) + 1)
    
    def score(self, query, doc_idx):
        tokens = self.tokenized[doc_idx]
        dl = len(tokens)
        tf_counter = Counter(tokens)
        total = 0.0
        for t in query.lower().split():
            tf = tf_counter.get(t, 0)
            numerator = tf * (self.k1 + 1)
            denominator = tf + (self.k1  * (1-self.b + self.b * dl / self.avgdl))
            total += self.idf(t) * numerator / denominator
            
        return total
            
    def search(self, query, top_k = 3):
        scores = [(i, self.score(query, i)) for i in range(self.N)]
        return sorted(scores, key=lambda x:x[1], reverse=True)[:top_k]
    

In [31]:
bm25 = BM25(documents)
for idx, score in bm25.search('Python 프로그래밍'):
    print(f"[{idx}] {score} | {documents[idx][:30]} ")

[0] 2.0360600223111596 | Python은 데이터 과학과 머신러닝에 널리 사용되는  
[1] 0.0 | 자연어 처리(NLP)는 컴퓨터가 인간의 언어를 이해하고 
[2] 0.0 | 벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고  


In [32]:
from langchain_community.retrievers import BM25Retriever

In [33]:
# from langchain.retrievers import BM25Retriever
# from langchain_classics import BM25Retriever

In [34]:
from langchain_core.documents import Document

In [35]:
docs_lc = [Document(page_content = d, metadata = {"index":i}) for i, d in enumerate(documents)]
bm25_retriever = BM25Retriever.from_documents(docs_lc)

In [36]:
bm25_retriever.k = 3

In [37]:
results = bm25_retriever.invoke('Python 프로그래밍')
results

[Document(metadata={'index': 0}, page_content='Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니다.'),
 Document(metadata={'index': 9}, page_content='임베딩은 텍스트를 수치 벡터로 변환하여 의미적 유사성을 측정할 수 있게 합니다.'),
 Document(metadata={'index': 8}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 입력을 설계하는 기술입니다.')]

In [38]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(docs_lc, embeddings)
vector_retriever = vectorstore.as_retriever(search_kwargs = {'k' : 3})

In [39]:
results = vector_retriever.invoke('딥러닝 모델 구조')
results

[Document(id='a94ddd35-9fd1-45da-9b51-7a2bb30e6adc', metadata={'index': 6}, page_content='트랜스포머 아키텍처는 어텐션 메커니즘을 활용한 딥러닝 모델 구조입니다.'),
 Document(id='f3ae9aa5-fd36-4550-beeb-0e760e19a9c8', metadata={'index': 3}, page_content='GPT-4는 OpenAI가 개발한 대규모 언어 모델로 다양한 작업을 수행합니다.'),
 Document(id='09b13d90-568f-448a-882c-901dcc6899ca', metadata={'index': 4}, page_content='RAG(검색 증강 생성)는 외부 지식을 활용하여 LLM 응답을 개선합니다.')]

In [40]:
from langchain_classic.retrievers import EnsembleRetriever

In [41]:
ensemble = EnsembleRetriever(
                retrievers = [vector_retriever, bm25_retriever],
                weights = [0.5, 0.5],
            )

In [42]:
results = ensemble.invoke("Python 데이터 과학")
results

[Document(id='ea501e00-ea33-48d3-9f16-f0f3ab55a1d0', metadata={'index': 0}, page_content='Python은 데이터 과학과 머신러닝에 널리 사용되는 프로그래밍 언어입니다.'),
 Document(id='fd7ec761-be33-4cc0-af20-aba33dd928de', metadata={'index': 9}, page_content='임베딩은 텍스트를 수치 벡터로 변환하여 의미적 유사성을 측정할 수 있게 합니다.'),
 Document(id='4c40e366-d31b-4c7b-b243-5db2c1343eba', metadata={'index': 2}, page_content='벡터 데이터베이스는 고차원 벡터를 효율적으로 저장하고 검색하는 시스템입니다.'),
 Document(metadata={'index': 8}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 입력을 설계하는 기술입니다.')]

In [43]:
for w_vec, w_bm25 in [(0.2, 0.8), (0.5, 0.5), (0.8, 0.2)]:
    ensemble = EnsembleRetriever(
                retrievers = [vector_retriever, bm25_retriever],
                weights = [w_vec, w_bm25],
            )
    results = ensemble.invoke("Python 데이터 과학")
    top_idx = results[0].metadata['index']
    print(f"BM25 = {w_bm25}, Vector = {w_vec} -> Top-1 : {top_idx}, {results[0].page_content[:30]}")

BM25 = 0.8, Vector = 0.2 -> Top-1 : 0, Python은 데이터 과학과 머신러닝에 널리 사용되는 
BM25 = 0.5, Vector = 0.5 -> Top-1 : 0, Python은 데이터 과학과 머신러닝에 널리 사용되는 
BM25 = 0.2, Vector = 0.8 -> Top-1 : 0, Python은 데이터 과학과 머신러닝에 널리 사용되는 


In [44]:
queries = ["Python 프로그래밍 언어", "딥러닝 모델 구조", "FAISS 라이브러리"]
for q in queries:
    bm25_res = bm25_retriever.invoke(q)
    vec_res = vector_retriever.invoke(q)
    ens_res = ensemble.invoke(q)
    
    bm25_ids = [d.metadata['index'] for d in bm25_res]
    vec_ids = [d.metadata['index'] for d in vec_res]
    ens_ids = [d.metadata['index'] for d in ens_res]
    
    print(q)
    print(f"BM25 : {bm25_ids}, vec : {vec_ids}, ens : {ens_ids}")

Python 프로그래밍 언어
BM25 : [0, 3, 9], vec : [0, 1, 3], ens : [0, 3, 1, 9]
딥러닝 모델 구조
BM25 : [6, 9, 8], vec : [6, 3, 4], ens : [6, 3, 4, 9, 8]
FAISS 라이브러리
BM25 : [9, 8, 7], vec : [7, 4, 2], ens : [7, 4, 2, 9, 8]
